## Project paths

In [10]:
# ==========================================================
# Project Path Setup
# ==========================================================

from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Project root could not be found. "
        "Run the notebook from the project root or notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project setup completed successfully.")
print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)

Project setup completed successfully.
Project root: /Users/mas364/Desktop/AI_Crypto_Forecasting
Raw data folder: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw


## Imports and output folders

In [11]:
# ==========================================================
# Imports and Google Trends Output Paths
# ==========================================================

from time import sleep
import random

import numpy as np
import pandas as pd
from pytrends.request import TrendReq

GOOGLE_TRENDS_WINDOWS_DIR = (
    RAW_DATA_DIR / "google_trends_windows"
)

GOOGLE_TRENDS_WINDOWS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GOOGLE_TRENDS_OUTPUT = (
    RAW_DATA_DIR / "google_trends.csv"
)

print("Raw windows folder:", GOOGLE_TRENDS_WINDOWS_DIR)
print("Final output file:", GOOGLE_TRENDS_OUTPUT)

Raw windows folder: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw/google_trends_windows
Final output file: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw/google_trends.csv


## Collection settings

In [12]:
# ==========================================================
# Google Trends Collection Settings
# ==========================================================

START_DATE = pd.Timestamp("2021-06-01")
END_DATE = pd.Timestamp("2026-05-06")

WINDOW_DAYS = 90
OVERLAP_DAYS = 30

GEO = ""

KEYWORDS = [
    "Bitcoin",
    "Ethereum"
]

print("Study period:", START_DATE.date(), "to", END_DATE.date())
print("Window length:", WINDOW_DAYS)
print("Overlap length:", OVERLAP_DAYS)

Study period: 2021-06-01 to 2026-05-06
Window length: 90
Overlap length: 30


## Create overlapping windows

In [13]:
# ==========================================================
# Create Overlapping Date Windows
# ==========================================================

def create_date_windows(
    start_date,
    end_date,
    window_days=90,
    overlap_days=30
):
    if overlap_days >= window_days:
        raise ValueError(
            "Overlap days must be smaller than window days."
        )

    windows = []

    current_start = pd.Timestamp(start_date)
    final_date = pd.Timestamp(end_date)

    step_days = window_days - overlap_days

    while current_start <= final_date:

        current_end = min(
            current_start + pd.Timedelta(days=window_days - 1),
            final_date
        )

        windows.append(
            (current_start, current_end)
        )

        if current_end == final_date:
            break

        current_start = (
            current_start
            + pd.Timedelta(days=step_days)
        )

    return windows


date_windows = create_date_windows(
    start_date=START_DATE,
    end_date=END_DATE,
    window_days=WINDOW_DAYS,
    overlap_days=OVERLAP_DAYS
)

print("Number of windows:", len(date_windows))

for number, (start, end) in enumerate(
    date_windows[:5],
    start=1
):
    print(
        number,
        start.date(),
        "to",
        end.date()
    )

Number of windows: 30
1 2021-06-01 to 2021-08-29
2 2021-07-31 to 2021-10-28
3 2021-09-29 to 2021-12-27
4 2021-11-28 to 2022-02-25
5 2022-01-27 to 2022-04-26


## Download one window

In [14]:
# ==========================================================
# Download One Keyword and One Window
# ==========================================================

def download_trends_window(
    keyword,
    start_date,
    end_date,
    geo="",
    max_retries=5
):
    timeframe = (
        f"{start_date.strftime('%Y-%m-%d')} "
        f"{end_date.strftime('%Y-%m-%d')}"
    )

    for attempt in range(1, max_retries + 1):

        try:
            pytrends = TrendReq(
                hl="en-GB",
                tz=0,
                timeout=(10, 30),
                retries=0,
                backoff_factor=0
            )

            pytrends.build_payload(
                kw_list=[keyword],
                timeframe=timeframe,
                geo=geo
            )

            window_data = (
                pytrends
                .interest_over_time()
            )

            if window_data.empty:
                raise ValueError(
                    f"No data returned for {keyword}, "
                    f"{timeframe}"
                )

            if "isPartial" in window_data.columns:
                window_data = window_data.drop(
                    columns=["isPartial"]
                )

            window_data = (
                window_data
                .reset_index()
                .rename(
                    columns={
                        "date": "Date",
                        keyword: "Search_Interest"
                    }
                )
            )

            window_data["Date"] = pd.to_datetime(
                window_data["Date"]
            )

            window_data["Search_Interest"] = pd.to_numeric(
                window_data["Search_Interest"],
                errors="coerce"
            )

            return window_data[
                ["Date", "Search_Interest"]
            ].copy()

        except Exception as error:

            print(
                f"Attempt {attempt}/{max_retries} failed "
                f"for {keyword}: {timeframe}"
            )

            print(
                f"Original error: "
                f"{type(error).__name__}: {error}"
            )

            if attempt == max_retries:
                raise RuntimeError(
                    f"Failed to download {keyword} "
                    f"for {timeframe}."
                ) from error

            wait_seconds = (
                attempt * 30
                + random.randint(10, 20)
            )

            print(
                f"Waiting {wait_seconds} seconds."
            )

            sleep(wait_seconds)

## Test one Bitcoin window

In [ ]:
# ==========================================================
# Test One Window
# ==========================================================

test_window = download_trends_window(
    keyword="Bitcoin",
    start_date=pd.Timestamp("2021-06-01"),
    end_date=pd.Timestamp("2021-08-29"),
    geo=GEO,
    max_retries=3
)

display(test_window.head())

print("Shape:", test_window.shape)

## Collect all windows for one keyword

In [18]:
def collect_keyword_windows(
    keyword,
    windows,
    output_folder,
    geo="",
    request_pause=(120, 180)
):
    """
    Collect all overlapping windows for one keyword.

    Existing CSV files are loaded and preserved.
    Only missing windows are downloaded.

    If Google returns a 429 rate-limit error, collection stops
    safely and can be resumed later.
    """

    keyword_folder = output_folder / keyword.lower()
    keyword_folder.mkdir(parents=True, exist_ok=True)

    collected_windows = []

    for window_number, (start_date, end_date) in enumerate(
        windows,
        start=1
    ):
        filename = (
            f"{keyword.lower()}_"
            f"{start_date.strftime('%Y%m%d')}_"
            f"{end_date.strftime('%Y%m%d')}.csv"
        )

        filepath = keyword_folder / filename

        # Reuse previously downloaded files
        if filepath.exists():
            print(
                f"{keyword}: window {window_number}/{len(windows)} "
                f"already downloaded, loading"
            )

            window_data = pd.read_csv(
                filepath,
                parse_dates=["Date"]
            )

            collected_windows.append(window_data)
            continue

        print(
            f"\n{keyword}: window {window_number}/{len(windows)} "
            f"({start_date.date()} to {end_date.date()})"
        )

        try:
            window_data = download_trends_window(
                keyword=keyword,
                start_date=start_date,
                end_date=end_date,
                geo=geo,
                max_retries=2
            )

        except RuntimeError as error:
            original_error = error.__cause__

            if (
                original_error is not None
                and "429" in str(original_error)
            ):
                print("\nGoogle rate limit reached.")
                print("Previously downloaded files are safe.")
                print("Stop now and resume later.")
                print("Next missing file:", filename)

                return collected_windows, False

            raise

        window_data.to_csv(
            filepath,
            index=False
        )

        collected_windows.append(window_data)

        print("Saved:", filepath)

        wait_seconds = random.uniform(
            request_pause[0],
            request_pause[1]
        )

        print(
            f"Waiting {wait_seconds:.0f} seconds "
            "before the next new request."
        )

        sleep(wait_seconds)

    print(f"\nAll {keyword} windows collected successfully.")

    return collected_windows, True

## Collect Bitcoin windows

In [20]:
# ==========================================================
# Collect Bitcoin Windows
# ==========================================================

bitcoin_windows, bitcoin_complete = collect_keyword_windows(
    keyword="Bitcoin",
    windows=date_windows,
    output_folder=GOOGLE_TRENDS_WINDOWS_DIR,
    geo=GEO,
    request_pause=(120, 180)
)

print("Bitcoin windows available:", len(bitcoin_windows))
print("Bitcoin collection complete:", bitcoin_complete)

Bitcoin: window 1/30 already downloaded, loading
Bitcoin: window 2/30 already downloaded, loading
Bitcoin: window 3/30 already downloaded, loading
Bitcoin: window 4/30 already downloaded, loading
Bitcoin: window 5/30 already downloaded, loading
Bitcoin: window 6/30 already downloaded, loading
Bitcoin: window 7/30 already downloaded, loading
Bitcoin: window 8/30 already downloaded, loading
Bitcoin: window 9/30 already downloaded, loading
Bitcoin: window 10/30 already downloaded, loading
Bitcoin: window 11/30 already downloaded, loading
Bitcoin: window 12/30 already downloaded, loading

Bitcoin: window 13/30 (2023-05-22 to 2023-08-19)
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw/google_trends_windows/bitcoin/bitcoin_20230522_20230819.csv
Waiting 139 seconds before the next new request.

Bitcoin: window 14/30 (2023-07-21 to 2023-10-18)
Saved: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw/google_trends_windows/bitcoin/bitcoin_20230721_20231018.csv
Waiting 157 seconds 

## Collect Ethereum windows

In [23]:
# ==========================================================
# Collect Ethereum Windows
# ==========================================================

ethereum_windows, ethereum_complete = collect_keyword_windows(
    keyword="Ethereum",
    windows=date_windows,
    output_folder=GOOGLE_TRENDS_WINDOWS_DIR,
    geo=GEO,
    request_pause=(60, 90)
)

print("Ethereum windows available:", len(ethereum_windows))
print("Ethereum collection complete:", ethereum_complete)

Ethereum: window 1/30 already downloaded, loading
Ethereum: window 2/30 already downloaded, loading
Ethereum: window 3/30 already downloaded, loading
Ethereum: window 4/30 already downloaded, loading
Ethereum: window 5/30 already downloaded, loading
Ethereum: window 6/30 already downloaded, loading
Ethereum: window 7/30 already downloaded, loading
Ethereum: window 8/30 already downloaded, loading
Ethereum: window 9/30 already downloaded, loading
Ethereum: window 10/30 already downloaded, loading
Ethereum: window 11/30 already downloaded, loading
Ethereum: window 12/30 already downloaded, loading
Ethereum: window 13/30 already downloaded, loading
Ethereum: window 14/30 already downloaded, loading
Ethereum: window 15/30 already downloaded, loading
Ethereum: window 16/30 already downloaded, loading
Ethereum: window 17/30 already downloaded, loading
Ethereum: window 18/30 already downloaded, loading
Ethereum: window 19/30 already downloaded, loading
Ethereum: window 20/30 already downloade

## Calculate overlap scale

In [24]:
# ==========================================================
# Calculate Overlap Scale Factor
# ==========================================================

def calculate_overlap_scale(
    stitched_data,
    new_window
):
    overlap = stitched_data.merge(
        new_window,
        on="Date",
        how="inner",
        suffixes=("_existing", "_new")
    )

    valid_overlap = overlap[
        (
            overlap["Search_Interest_existing"] > 0
        )
        & (
            overlap["Search_Interest_new"] > 0
        )
    ].copy()

    if valid_overlap.empty:
        raise ValueError(
            "No valid positive overlap was found."
        )

    ratios = (
        valid_overlap["Search_Interest_existing"]
        / valid_overlap["Search_Interest_new"]
    )

    scale_factor = ratios.median()

    if (
        not np.isfinite(scale_factor)
        or scale_factor <= 0
    ):
        raise ValueError(
            f"Invalid scale factor: {scale_factor}"
        )

    return float(scale_factor), len(valid_overlap)

## Stitch all windows

In [25]:
# ==========================================================
# Stitch Overlapping Windows
# ==========================================================

def stitch_keyword_windows(
    window_list
):
    if not window_list:
        raise ValueError(
            "No Google Trends windows were provided."
        )

    stitched = (
        window_list[0]
        .copy()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    scaling_log = []

    for window_number, new_window in enumerate(
        window_list[1:],
        start=2
    ):
        new_window = (
            new_window
            .copy()
            .sort_values("Date")
            .reset_index(drop=True)
        )

        scale_factor, overlap_days = (
            calculate_overlap_scale(
                stitched,
                new_window
            )
        )

        new_window["Search_Interest"] = (
            new_window["Search_Interest"]
            * scale_factor
        )

        stitched = pd.concat(
            [stitched, new_window],
            ignore_index=True
        )

        stitched = (
            stitched
            .groupby(
                "Date",
                as_index=False
            )["Search_Interest"]
            .mean()
            .sort_values("Date")
            .reset_index(drop=True)
        )

        scaling_log.append({
            "Window_Number": window_number,
            "Scale_Factor": scale_factor,
            "Valid_Overlap_Days": overlap_days
        })

    scaling_log = pd.DataFrame(
        scaling_log
    )

    return stitched, scaling_log

## Stitch Bitcoin

In [26]:
# ==========================================================
# Stitch Bitcoin Windows
# ==========================================================

bitcoin_stitched, bitcoin_scaling_log = (
    stitch_keyword_windows(
        bitcoin_windows
    )
)

display(bitcoin_stitched.head())
display(bitcoin_scaling_log.head())

print(
    "Bitcoin stitched rows:",
    len(bitcoin_stitched)
)

,Date,Search_Interest
0,2021-06-01,69.0
1,2021-06-02,68.0
2,2021-06-03,68.0
3,2021-06-04,76.0
4,2021-06-05,68.0


,Window_Number,Scale_Factor,Valid_Overlap_Days
0,2,0.841270,30
1,3,0.985382,30
2,4,2.178999,30
3,5,0.907916,30
4,6,1.162133,30


Bitcoin stitched rows: 1801


## Stitch Ethereum

In [27]:
# ==========================================================
# Stitch Ethereum Windows
# ==========================================================

ethereum_stitched, ethereum_scaling_log = (
    stitch_keyword_windows(
        ethereum_windows
    )
)

display(ethereum_stitched.head())
display(ethereum_scaling_log.head())

print(
    "Ethereum stitched rows:",
    len(ethereum_stitched)
)

,Date,Search_Interest
0,2021-06-01,83.0
1,2021-06-02,80.0
2,2021-06-03,80.0
3,2021-06-04,76.0
4,2021-06-05,68.0


,Window_Number,Scale_Factor,Valid_Overlap_Days
0,2,1.000000,30
1,3,0.966479,30
2,4,2.431678,30
3,5,0.786169,30
4,6,1.004085,30


Ethereum stitched rows: 1801


## Normalise one complete series

In [28]:
# ==========================================================
# Normalise Final Series to 0–100
# ==========================================================

def normalise_to_100(
    data,
    value_column="Search_Interest"
):
    result = data.copy()

    maximum = result[value_column].max()

    if (
        pd.isna(maximum)
        or maximum <= 0
    ):
        raise ValueError(
            f"Cannot normalise the series. "
            f"Maximum value: {maximum}"
        )

    result[value_column] = (
        result[value_column]
        / maximum
        * 100
    )

    result[value_column] = (
        result[value_column]
        .round()
        .clip(lower=0, upper=100)
        .astype("Int64")
    )

    return result

## Normalise Bitcoin and Ethereum independently

In [29]:
# ==========================================================
# Independently Normalise Both Series
# ==========================================================

bitcoin_final = (
    normalise_to_100(
        bitcoin_stitched
    )
    .rename(
        columns={
            "Search_Interest":
            "Bitcoin"
        }
    )
)

ethereum_final = (
    normalise_to_100(
        ethereum_stitched
    )
    .rename(
        columns={
            "Search_Interest":
            "Ethereum"
        }
    )
)

display(bitcoin_final.head())
display(ethereum_final.head())

,Date,Bitcoin
0,2021-06-01,32
1,2021-06-02,31
2,2021-06-03,31
3,2021-06-04,35
4,2021-06-05,31


,Date,Ethereum
0,2021-06-01,34
1,2021-06-02,33
2,2021-06-03,33
3,2021-06-04,31
4,2021-06-05,28


## Merge the final series

In [30]:
# ==========================================================
# Merge Bitcoin and Ethereum Search Series
# ==========================================================

google_trends = bitcoin_final.merge(
    ethereum_final,
    on="Date",
    how="inner",
    validate="one_to_one"
)

google_trends = (
    google_trends
    .sort_values("Date")
    .reset_index(drop=True)
)

display(google_trends.head())
display(google_trends.tail())

print("Shape:", google_trends.shape)

,Date,Bitcoin,Ethereum
0,2021-06-01,32,34
1,2021-06-02,31,33
2,2021-06-03,31,33
3,2021-06-04,35,31
4,2021-06-05,31,28


,Date,Bitcoin,Ethereum
1796,2026-05-02,14,9
1797,2026-05-03,14,9
1798,2026-05-04,16,9
1799,2026-05-05,17,11
1800,2026-05-06,20,12


Shape: (1801, 3)


## Save final data and scaling logs

In [32]:
# ==========================================================
# Save Final Google Trends Dataset
# ==========================================================

google_trends.to_csv(
    GOOGLE_TRENDS_OUTPUT,
    index=False
)

bitcoin_scaling_log.to_csv(
    GOOGLE_TRENDS_WINDOWS_DIR
    / "bitcoin_scaling_log.csv",
    index=False
)

ethereum_scaling_log.to_csv(
    GOOGLE_TRENDS_WINDOWS_DIR
    / "ethereum_scaling_log.csv",
    index=False
)

print(
    "Google Trends dataset saved successfully."
)

print(
    "Final dataset:",
    GOOGLE_TRENDS_OUTPUT
)

print(
    "Rows:",
    google_trends.shape[0]
)

print(
    "Columns:",
    google_trends.shape[1]
)

Google Trends dataset saved successfully.
Final dataset: /Users/mas364/Desktop/AI_Crypto_Forecasting/data/raw/google_trends.csv
Rows: 1801
Columns: 3


## Final quality checks

In [33]:
# ==========================================================
# Final Google Trends Validation
# ==========================================================

expected_dates = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

missing_dates = (
    expected_dates
    .difference(
        google_trends["Date"]
    )
)

duplicate_dates = (
    google_trends["Date"]
    .duplicated()
    .sum()
)

validation_summary = pd.DataFrame({
    "Check": [
        "Rows",
        "Duplicate dates",
        "Missing dates",
        "Missing Bitcoin values",
        "Missing Ethereum values",
        "Bitcoin minimum",
        "Bitcoin maximum",
        "Ethereum minimum",
        "Ethereum maximum"
    ],
    "Result": [
        len(google_trends),
        duplicate_dates,
        len(missing_dates),
        google_trends["Bitcoin"].isna().sum(),
        google_trends["Ethereum"].isna().sum(),
        google_trends["Bitcoin"].min(),
        google_trends["Bitcoin"].max(),
        google_trends["Ethereum"].min(),
        google_trends["Ethereum"].max()
    ]
})

display(validation_summary)

if duplicate_dates > 0:
    raise ValueError(
        "Duplicate dates were found."
    )

if google_trends[
    ["Bitcoin", "Ethereum"]
].isna().any().any():
    raise ValueError(
        "Missing search values were found."
    )

if not google_trends[
    "Bitcoin"
].between(0, 100).all():
    raise ValueError(
        "Bitcoin values are outside 0–100."
    )

if not google_trends[
    "Ethereum"
].between(0, 100).all():
    raise ValueError(
        "Ethereum values are outside 0–100."
    )

print(
    "Google Trends validation completed successfully."
)

,Check,Result
0,Rows,1801
1,Duplicate dates,0
2,Missing dates,0
3,Missing Bitcoin values,0
4,Missing Ethereum values,0
5,Bitcoin minimum,8
6,Bitcoin maximum,100
7,Ethereum minimum,4
8,Ethereum maximum,100


Google Trends validation completed successfully.
